COVID-19 Analytics (AWS) — End-to-End
 


In [0]:
# Silver curation

from pyspark.sql import SparkSession, functions as F

BUCKET = "YOUR_BUCKET"
BRONZE = f"s3://{BUCKET}/covid/bronze"
SILVER = f"s3://{BUCKET}/covid/silver"

spark = SparkSession.builder.appName("covid-minimal-silver").getOrCreate()

# Helpers
upper_trim = lambda c: F.upper(F.trim(F.col(c)))
to_date = lambda c: F.to_date(F.col(c).cast("string"))


# --- Lookups ---
states = (spark.read.option("header", True).csv(f"{BRONZE}/static/states_abv.csv")
          .select(upper_trim("abbr").alias("state_code"), F.initcap("name").alias("state_name")))

# --- Cases (NYT state file: date,state,cases,deaths or JHU state-level variant) ---
cases_raw = spark.read.option("header", True).csv(f"{BRONZE}/nytimes/us_states.csv")

cases_std = (cases_raw
  # Handle date formats like "2020-05-01" or yyyymmdd; adjust if needed
  .withColumn("full_date", F.to_date("date"))
  # If the file stores state NAME, map to code; if it already stores code, this join still works
  .withColumn("state_name_raw", F.initcap(F.col("state")))
  .join(states, states.state_name == F.col("state_name_raw"), "left")
  .withColumn("cases_cum", F.col("cases").cast("long"))
  .withColumn("deaths_cum", F.col("deaths").cast("long"))
  .withColumn("year", F.year("full_date"))
  .withColumn("month", F.month("full_date"))
  .withColumn("day", F.dayofmonth("full_date"))
  .select("full_date", "state_code", "state_name", "cases_cum", "deaths_cum", "year", "month", "day")
  .dropna(subset=["full_date", "state_code"])
)

(cases_std.write.mode("overwrite")
  .partitionBy("state_code", "year", "month", "day")
  .parquet(f"{SILVER}/cases_standardized"))
